# Rage-Click & UI Friction Detection — Exploratory Data Analysis

This notebook provides an in-depth exploratory analysis of client-side interaction telemetry and the 5 engineered behavioral features used to train the Feed-Forward Neural Network (FFNN).

### Five Numerical Features:
1. **`click_frequency`**: Total clicks divided by the 5-second window duration (clicks/sec).
2. **`rapid_fire_clicks`**: Consecutive clicks on the exact same DOM element within 300 ms.
3. **`maximum_cursor_velocity`**: Peak velocity across consecutive cursor positions (px/sec).
4. **`erratic_direction_changes`**: Significant movement direction shifts (> 90°) above the jitter threshold (5px).
5. **`scroll_thrashing`**: Total cumulative vertical distance traveled during directional reversals within 2.0s.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from src.utils import load_config
config = load_config(PROJECT_ROOT / "config.yaml")
print("Configuration loaded successfully.")

## 1. Load Processed Dataset
We load the labeled interaction dataset generated by `scripts/train_model.py`.

In [2]:
data_path = PROJECT_ROOT / "data" / "processed" / "labeled_features.csv"
if not data_path.exists():
    data_path = PROJECT_ROOT / "data" / "synthetic" / "synthetic_features.csv"

df = pd.read_csv(data_path)
print(f"Loaded {len(df):,} interaction windows.")
df.head()

## 2. Descriptive Statistics by Behavioral Class

In [3]:
features = config["features"]
label_col = "label" if "label" in df.columns else "ground_truth_label"
df.groupby(label_col)[features].describe().T

## 3. Correlation Matrix
Examine inter-feature collinearity and relationships.

In [4]:
plt.figure(figsize=(8, 6))
corr = df[features].corr()
sns.heatmap(corr, annot=True, cmap="coolwarm", vmin=-1, vmax=1, fmt=".2f", square=True)
plt.title("Feature Correlation Matrix", fontsize=14, weight="bold")
plt.tight_layout()
plt.show()

## 4. Feature Distributions: Normal vs Friction (Class Overlap Analysis)
Notice that the distributions contain intentional overlap. Some normal sessions exhibit high velocity or fast double clicks, while some frustrated sessions manifest purely as scroll thrashing without rage clicks.

In [5]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.ravel()

for i, feat in enumerate(features):
    sns.kdeplot(data=df, x=feat, hue=label_col, common_norm=False, fill=True, ax=axes[i], palette=["#3498db", "#e74c3c"])
    axes[i].set_title(f"Distribution: {feat}", weight="bold")
    axes[i].grid(True, linestyle=":", alpha=0.6)

sns.countplot(data=df, x=label_col, ax=axes[5], palette=["#3498db", "#e74c3c"])
axes[5].set_title("Class Imbalance (Normal vs Friction)", weight="bold")
axes[5].set_xticklabels(["Normal (0)", "Friction (1)"])
plt.tight_layout()
plt.show()

## 5. Live Inference Verification
Load the trained model and verify predictions on sample friction and normal vectors.

In [6]:
from src.inference import get_inference_engine

engine = get_inference_engine()

sample_normal = {
    "click_frequency": 0.4,
    "rapid_fire_clicks": 0,
    "maximum_cursor_velocity": 550.0,
    "erratic_direction_changes": 1,
    "scroll_thrashing": 0.0,
}

sample_rage = {
    "click_frequency": 3.8,
    "rapid_fire_clicks": 6,
    "maximum_cursor_velocity": 2400.0,
    "erratic_direction_changes": 7,
    "scroll_thrashing": 950.0,
}

pred_normal = engine.predict(sample_normal)
pred_rage = engine.predict(sample_rage)

print("Sample Normal Interaction:", pred_normal["friction_probability"], "-> Friction Detected?", pred_normal["friction_detected"])
print("Sample Rage Interaction:  ", pred_rage["friction_probability"], "-> Friction Detected?", pred_rage["friction_detected"])
